# Construindo banco de dados das amostras de vigas

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import itertools
import numpy as np
from parepy_toolbox import sampling_algorithm_structural_analysis
from obj import momento_limite_armadura_simples, area_aco_flexao_simples, momento_resistente_secao_sem_cor, obj_mestrado_victor, verifica_tempo_limite

# Planejamento experimental

In [2]:
# Definindo os níveis das variáveis
divs = 5
n_levels = 5
n_times = 5
h_levels = np.linspace(0.30, 0.70, n_levels)
b_levels = np.linspace(0.14, 0.35, n_levels)
f_levels = np.linspace(20000, 50000, n_levels)
combinacoes = list(itertools.product(h_levels, b_levels, f_levels))
df = pd.DataFrame(combinacoes, columns=['h', 'b_w', 'f_ck'])
df

,h,b_w,f_ck
0,0.3,0.14,20000.0
1,0.3,0.14,27500.0
2,0.3,0.14,35000.0
3,0.3,0.14,42500.0
4,0.3,0.14,50000.0
...,...,...,...
120,0.7,0.35,20000.0
121,0.7,0.35,27500.0
122,0.7,0.35,35000.0
123,0.7,0.35,42500.0


# Criando o banco de armaduras

In [3]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
for i in range(len(df)):
    b_w.append(df['b_w'][i])
    h.append(df['h'][i])
    f_ck.append(df['f_ck'][i])
    m_rdlim.append(momento_limite_armadura_simples(df['b_w'][i], df['h'][i], df['f_ck'][i]))
    a_saux, pho_saux = area_aco_flexao_simples(m_rdlim[i], df['b_w'][i], df['h'][i], df['f_ck'][i])
    pho_s.append(pho_saux)
    a_s.append(a_saux)

df['m_rdlim'] = m_rdlim
df['a_s'] = a_s
df['pho_s'] = pho_s
df

,h,b_w,f_ck,m_rdlim,a_s,pho_s
0,0.3,0.14,20000.0,36.584136,0.000380,0.904886
1,0.3,0.14,27500.0,50.303187,0.000523,1.244218
2,0.3,0.14,35000.0,64.022238,0.000665,1.583550
3,0.3,0.14,42500.0,77.741289,0.000808,1.922882
4,0.3,0.14,50000.0,91.460340,0.000950,2.262214
...,...,...,...,...,...,...
120,0.7,0.35,20000.0,497.950740,0.002217,0.904886
121,0.7,0.35,27500.0,684.682268,0.003048,1.244218
122,0.7,0.35,35000.0,871.413795,0.003880,1.583550
123,0.7,0.35,42500.0,1058.145323,0.004711,1.922882


# Subdividindo em grupos por classe de densidade de armadura

In [4]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
for indece, linha in df.iterrows():
    aux = linha['pho_s'] / divs
    pho = 0
    for i in range(divs):
        pho += aux
        pho_s.append(pho)
        b_w.append(linha['b_w'])
        h.append(linha['h'])
        f_ck.append(linha['f_ck'])
        m_rdlim.append(linha['m_rdlim'])
        a_s.append(pho * linha['b_w'] * linha['h'] / 100)

df_aux = {'b_w': b_w, 'h': h, 'f_ck': f_ck, 'm_rdlim': m_rdlim, 'a_s': a_s, 'pho_s': pho_s}
df_aux = pd.DataFrame(df_aux)
df_aux['m_rd'] = df_aux.apply(lambda row: momento_resistente_secao_sem_cor(row['a_s'], row['b_w'], row['h'],row['f_ck']), axis=1)
df_aux

,b_w,h,f_ck,m_rdlim,a_s,pho_s,m_rd
0,0.14,0.3,20000.0,36.584136,0.000076,0.180977,9.836582
1,0.14,0.3,20000.0,36.584136,0.000152,0.361954,18.823519
2,0.14,0.3,20000.0,36.584136,0.000228,0.542931,26.960813
3,0.14,0.3,20000.0,36.584136,0.000304,0.723909,34.248462
4,0.14,0.3,20000.0,36.584136,0.000380,0.904886,40.686467
...,...,...,...,...,...,...,...
620,0.35,0.7,50000.0,1244.876850,0.001108,0.452443,334.717022
621,0.35,0.7,50000.0,1244.876850,0.002217,0.904886,640.522538
622,0.35,0.7,50000.0,1244.876850,0.003325,1.357329,917.416549
623,0.35,0.7,50000.0,1244.876850,0.004434,1.809771,1165.399054


In [5]:
df_aux.drop(['m_rdlim'], axis='columns', inplace=True)
df_aux

,b_w,h,f_ck,a_s,pho_s,m_rd
0,0.14,0.3,20000.0,0.000076,0.180977,9.836582
1,0.14,0.3,20000.0,0.000152,0.361954,18.823519
2,0.14,0.3,20000.0,0.000228,0.542931,26.960813
3,0.14,0.3,20000.0,0.000304,0.723909,34.248462
4,0.14,0.3,20000.0,0.000380,0.904886,40.686467
...,...,...,...,...,...,...
620,0.35,0.7,50000.0,0.001108,0.452443,334.717022
621,0.35,0.7,50000.0,0.002217,0.904886,640.522538
622,0.35,0.7,50000.0,0.003325,1.357329,917.416549
623,0.35,0.7,50000.0,0.004434,1.809771,1165.399054


# Análise de confiabilidade no tempo

In [6]:
tempos = list(range(0, 101, n_times))
beta = []
b_w_aux_list = []
h_aux_list = []
f_ck_aux_list = []
m_rd_aux_list = []
a_s_aux_list = []
mgk_aux_list = []
mqk_aux_list = []
time_aux_list = []
for i, row in df_aux.iterrows():
    b_w_aux = row['b_w']
    h_aux = row['h']
    f_ck_aux = row['f_ck']
    m_rd_aux = row['m_rd']
    a_s_aux = row['a_s']
    chi_list = list(np.linspace(0.10, 0.60, divs, endpoint=True))
    gamma_g = 1.40
    gamma_q = 1.40
    dados_viga = {'h (m)': h_aux, 'b_w (m)': b_w_aux, 'm_rd (kN.m)': m_rd_aux, 'a_s (m2)': a_s_aux, 'gamma_c': 1.00, 'gamma_s': 1.00, 'gamma_f': 1.00}
    none_variable = {'dados_viga': dados_viga, 'time analysis': tempos}

    for id, chi in enumerate(chi_list):
        den_g = gamma_g + gamma_q*chi/(1-chi)
        den_q = gamma_g*(1-chi)/chi + gamma_q
        m_gk = m_rd_aux/den_g
        m_qk = m_rd_aux/den_q

        # Data
        g = {'type': 'normal', 'parameters': {'mean': 1.06*m_gk, 'sigma': 0.12*1.06*m_gk}, 'stochastic variable': False}
        q = {'type': 'gumbel max', 'parameters': {'mean': 0.21*m_qk, 'sigma': 0.21*0.76*m_qk}, 'stochastic variable': True}
        f_ck = {'type': 'normal', 'parameters': {'mean': 1.22*f_ck_aux, 'sigma': 0.15*1.22*f_ck_aux}, 'stochastic variable': False}
        f_yk = {'type': 'normal', 'parameters': {'mean': 1.22*500000, 'sigma': 0.04*1.22*500000}, 'stochastic variable': False}
        teta_r = {'type': 'normal', 'parameters': {'mean': 1, 'sigma': 0.05}, 'stochastic variable': False}
        teta_s = {'type': 'normal', 'parameters': {'mean': 1, 'sigma': 0.05}, 'stochastic variable': False}
        var = [g, q, f_ck, f_yk, teta_r, teta_s]

        # PAREpy setup
        setup = {
                    'number of samples': 40000,
                    'numerical model': {'model sampling': 'lhs-time', 'time steps': len(none_variable['time analysis'])},
                    'variables settings': var,
                    'number of state limit functions or constraints': 1,
                    'none variable': none_variable,
                    'objective function': obj_mestrado_victor,
                    'name simulation': None,
                }
        # Call algorithm
        results_aux, pf_aux, beta_aux = sampling_algorithm_structural_analysis(setup)
        pf_aux_aux = pf_aux['G_0'].tolist()
        time_limit = verifica_tempo_limite(pf_aux_aux, tempos, pf_limit=6.87138E-4)
        b_w_aux_list.append(b_w_aux)
        h_aux_list.append(h_aux)
        f_ck_aux_list.append(f_ck_aux)
        m_rd_aux_list.append(m_rd_aux)
        mgk_aux_list.append(m_gk)
        mqk_aux_list.append(m_qk)
        a_s_aux_list.append(a_s_aux)
        time_aux_list.append(time_limit)
        
        # Supondo que você tenha um DataFrame chamado df_aux
        #results_aux.to_excel("df_aux.xlsx", index=False)

final_df = {
            'b_w': b_w_aux_list,
            'h': h_aux_list,
            'f_ck': f_ck_aux_list,
            'm_rd': m_rd_aux_list,
            'm_gk': mgk_aux_list,
            'm_qk': mqk_aux_list,
            'a_s': a_s_aux_list,
            'time': time_aux_list,
           }
final_df = pd.DataFrame(final_df)


00:08:37 - Checking inputs completed!
00:08:37 - Started State Limit Function evaluation (g)...
00:08:51 - Finished State Limit Function evaluation (g) in 1.40e+01 seconds!
00:08:51 - Started evaluation beta reliability index and failure probability...
00:08:51 - Finished evaluation beta reliability index and failure probability in 7.11e-02 seconds!
00:08:51 - Voilà!!!!....simulation results were not saved in a text file!
00:08:51 - Checking inputs completed!
00:08:51 - Started State Limit Function evaluation (g)...
00:09:05 - Finished State Limit Function evaluation (g) in 1.40e+01 seconds!
00:09:05 - Started evaluation beta reliability index and failure probability...
00:09:06 - Finished evaluation beta reliability index and failure probability in 1.38e-01 seconds!
00:09:06 - Voilà!!!!....simulation results were not saved in a text file!
00:09:06 - Checking inputs completed!
00:09:06 - Started State Limit Function evaluation (g)...
00:09:19 - Finished State Limit Function evaluation 

In [7]:
final_df.to_excel("final_df_tempo_limite.xlsx", index=False)

In [8]:
final_df.head(20)

,b_w,h,f_ck,m_rd,m_gk,m_qk,a_s,time
0,0.14,0.3,20000.0,9.836582,6.323517,0.702613,0.000076,47.180765
1,0.14,0.3,20000.0,9.836582,5.445251,1.580879,0.000076,64.788961
2,0.14,0.3,20000.0,9.836582,4.566984,2.459145,0.000076,72.239461
3,0.14,0.3,20000.0,9.836582,3.688718,3.337412,0.000076,76.586850
4,0.14,0.3,20000.0,9.836582,2.810452,4.215678,0.000076,78.950605
5,0.14,0.3,20000.0,18.823519,12.100834,1.344537,0.000152,52.053289
6,0.14,0.3,20000.0,18.823519,10.420163,3.025208,0.000152,64.496821
7,0.14,0.3,20000.0,18.823519,8.739491,4.705880,0.000152,73.799074
8,0.14,0.3,20000.0,18.823519,7.058820,6.386551,0.000152,78.193143
9,0.14,0.3,20000.0,18.823519,5.378148,8.067223,0.000152,80.940341
